# Zomato Bangalore Restaurant Analytics
## Notebook 2: SQL Analysis

This notebook runs 15 SQL queries against the Zomato SQLite database and prints the results.

> **Prerequisite**: Run `01_data_loading_and_cleaning.ipynb` first to create the database.

---

In [ ]:
# ============================================================
# IMPORTS & DATABASE CONNECTION
# ============================================================
import pandas as pd
import sqlite3
import warnings
warnings.filterwarnings('ignore')

DB_PATH = '../zomato_bangalore.db'
conn = sqlite3.connect(DB_PATH)

def run_query(sql, title=''):
    """Execute SQL query and return a styled DataFrame."""
    if title:
        print(f'\n{"=" * 60}')
        print(f'  {title}')
        print(f'{"=" * 60}')
    result = pd.read_sql_query(sql, conn)
    return result

# Verify tables exist
tables = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table'", conn)
print(f'Connected to database. Tables: {list(tables["name"])}')

---
## Section A: Descriptive Analytics (Q1–Q5)

In [ ]:
# -------------------------------------------------------
# Q1: Distribution of ratings across Bangalore
# SQL: CASE WHEN, GROUP BY, COUNT, Window function for %
# -------------------------------------------------------
q1 = run_query('''
SELECT
    CASE
        WHEN rating >= 1.0 AND rating < 2.0 THEN '1.0 – 1.9'
        WHEN rating >= 2.0 AND rating < 3.0 THEN '2.0 – 2.9'
        WHEN rating >= 3.0 AND rating < 3.5 THEN '3.0 – 3.4'
        WHEN rating >= 3.5 AND rating < 4.0 THEN '3.5 – 3.9'
        WHEN rating >= 4.0 AND rating < 4.5 THEN '4.0 – 4.4'
        WHEN rating >= 4.5 AND rating <= 5.0 THEN '4.5 – 5.0'
    END AS rating_bucket,
    COUNT(*) AS restaurant_count,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) AS percentage
FROM restaurants
WHERE rating IS NOT NULL
GROUP BY rating_bucket
ORDER BY rating_bucket
''', 'Q1: Rating Distribution')

q1

In [ ]:
# -------------------------------------------------------
# Q2: Top 20 most popular cuisines by restaurant count
# SQL: JOIN, GROUP BY, ORDER BY, LIMIT
# -------------------------------------------------------
q2 = run_query('''
SELECT
    cuisine,
    COUNT(*) AS restaurant_count,
    ROUND(AVG(r.rating), 2) AS avg_rating,
    ROUND(AVG(r.approx_cost), 0) AS avg_cost
FROM restaurant_cuisines rc
JOIN restaurants r ON r.id = rc.restaurant_id
WHERE r.rating IS NOT NULL
GROUP BY cuisine
ORDER BY restaurant_count DESC
LIMIT 20
''', 'Q2: Top 20 Cuisines')

q2

In [ ]:
# -------------------------------------------------------
# Q3: Top 25 locations by restaurant count
# SQL: GROUP BY, COUNT, AVG, ORDER BY
# -------------------------------------------------------
q3 = run_query('''
SELECT
    location,
    COUNT(*) AS restaurant_count,
    ROUND(AVG(rating), 2) AS avg_rating,
    ROUND(AVG(approx_cost), 0) AS avg_cost
FROM restaurants
WHERE location IS NOT NULL AND location != ''
GROUP BY location
ORDER BY restaurant_count DESC
LIMIT 25
''', 'Q3: Top 25 Locations')

q3

In [ ]:
# -------------------------------------------------------
# Q4: Avg cost by restaurant type
# SQL: AVG, MIN, MAX, GROUP BY, HAVING
# -------------------------------------------------------
q4 = run_query('''
SELECT
    rest_type,
    COUNT(*) AS count,
    ROUND(AVG(approx_cost), 0) AS avg_cost,
    ROUND(AVG(rating), 2) AS avg_rating,
    MIN(approx_cost) AS min_cost,
    MAX(approx_cost) AS max_cost
FROM restaurants
WHERE rest_type IS NOT NULL AND approx_cost IS NOT NULL
GROUP BY rest_type
HAVING COUNT(*) >= 50
ORDER BY avg_cost DESC
''', 'Q4: Avg Cost by Restaurant Type')

q4

In [ ]:
# -------------------------------------------------------
# Q5: Online order availability %
# SQL: CASE, percentage calculation with subquery
# -------------------------------------------------------
q5 = run_query('''
SELECT
    CASE WHEN online_order = 1 THEN 'Online Order Available'
         ELSE 'No Online Order'
    END AS order_status,
    COUNT(*) AS restaurant_count,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM restaurants), 2) AS percentage
FROM restaurants
GROUP BY online_order
''', 'Q5: Online Order Availability')

q5

---
## Section B: Comparative & Diagnostic Analytics (Q6–Q10)

In [ ]:
# -------------------------------------------------------
# Q6: Online orders vs rating
# SQL: Conditional aggregation, GROUP BY
# -------------------------------------------------------
q6 = run_query('''
SELECT
    CASE WHEN online_order = 1 THEN 'Yes' ELSE 'No' END AS online_order,
    COUNT(*) AS count,
    ROUND(AVG(rating), 3) AS avg_rating,
    ROUND(AVG(votes), 0) AS avg_votes,
    ROUND(AVG(approx_cost), 0) AS avg_cost
FROM restaurants
WHERE rating IS NOT NULL
GROUP BY online_order
''', 'Q6: Online Order Impact on Rating')

q6

In [ ]:
# -------------------------------------------------------
# Q7: Table booking vs rating & cost
# SQL: Same pattern, different dimension
# -------------------------------------------------------
q7 = run_query('''
SELECT
    CASE WHEN book_table = 1 THEN 'Yes' ELSE 'No' END AS table_booking,
    COUNT(*) AS count,
    ROUND(AVG(rating), 3) AS avg_rating,
    ROUND(AVG(approx_cost), 0) AS avg_cost,
    ROUND(AVG(votes), 0) AS avg_votes
FROM restaurants
WHERE rating IS NOT NULL
GROUP BY book_table
''', 'Q7: Table Booking Impact')

q7

In [ ]:
# -------------------------------------------------------
# Q8: Top 10 most-voted restaurants
# SQL: ORDER BY on multiple columns
# -------------------------------------------------------
q8 = run_query('''
SELECT
    name,
    location,
    rest_type,
    votes,
    rating,
    approx_cost
FROM restaurants
WHERE rating IS NOT NULL
ORDER BY votes DESC
LIMIT 10
''', 'Q8: Top 10 Most-Voted Restaurants')

q8

In [ ]:
# -------------------------------------------------------
# Q9: Overpriced cuisines (highest cost/rating ratio)
# SQL: AVG ratio, HAVING, NULLIF
# -------------------------------------------------------
q9 = run_query('''
SELECT
    rc.cuisine,
    COUNT(*) AS restaurant_count,
    ROUND(AVG(r.approx_cost), 0) AS avg_cost,
    ROUND(AVG(r.rating), 2) AS avg_rating,
    ROUND(AVG(r.approx_cost) / NULLIF(AVG(r.rating), 0), 0) AS cost_per_rating_point
FROM restaurant_cuisines rc
JOIN restaurants r ON r.id = rc.restaurant_id
WHERE r.rating IS NOT NULL AND r.approx_cost IS NOT NULL
GROUP BY rc.cuisine
HAVING COUNT(*) >= 30
ORDER BY cost_per_rating_point DESC
LIMIT 15
''', 'Q9: Overpriced Cuisines')

q9

In [ ]:
# -------------------------------------------------------
# Q10: Market saturation by location
# SQL: Composite scoring, HAVING
# -------------------------------------------------------
q10 = run_query('''
SELECT
    location,
    COUNT(*) AS restaurant_count,
    ROUND(AVG(rating), 2) AS avg_rating,
    ROUND(AVG(approx_cost), 0) AS avg_cost,
    ROUND(COUNT(*) * 1.0 / NULLIF(AVG(rating), 0), 2) AS saturation_score
FROM restaurants
WHERE rating IS NOT NULL AND location IS NOT NULL
GROUP BY location
HAVING COUNT(*) >= 50
ORDER BY saturation_score DESC
LIMIT 15
''', 'Q10: Market Saturation')

q10

---
## Section C: Advanced SQL Queries (Q11–Q15)

In [ ]:
# -------------------------------------------------------
# Q11: Top 3 restaurants per location (Window: RANK)
# SQL: RANK() OVER (PARTITION BY ... ORDER BY ...)
# -------------------------------------------------------
q11 = run_query('''
WITH ranked AS (
    SELECT
        name,
        location,
        rating,
        votes,
        approx_cost,
        RANK() OVER (
            PARTITION BY location
            ORDER BY rating DESC, votes DESC
        ) AS rank_in_location
    FROM restaurants
    WHERE rating IS NOT NULL AND location IS NOT NULL
)
SELECT * FROM ranked
WHERE rank_in_location <= 3
ORDER BY location, rank_in_location
''', 'Q11: Top 3 Restaurants per Location (RANK Window Function)')

# Show first 30 rows
q11.head(30)

In [ ]:
# -------------------------------------------------------
# Q12: Running average of cost (Window: AVG + ROWS)
# SQL: Nested window function on aggregated result
# -------------------------------------------------------
q12 = run_query('''
SELECT
    location,
    COUNT(*) AS restaurant_count,
    ROUND(AVG(approx_cost), 0) AS avg_cost,
    ROUND(
        AVG(AVG(approx_cost)) OVER (
            ORDER BY location
            ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
        ), 0
    ) AS running_avg_3
FROM restaurants
WHERE approx_cost IS NOT NULL AND location IS NOT NULL AND location != ''
GROUP BY location
ORDER BY location
''', 'Q12: Running Average of Cost by Location')

q12.head(20)

In [ ]:
# -------------------------------------------------------
# Q13: Hidden gems
# SQL: Compound WHERE with CASE categorization
# -------------------------------------------------------
q13 = run_query('''
SELECT
    name,
    location,
    rest_type,
    rating,
    votes,
    approx_cost,
    CASE
        WHEN rating >= 4.5 AND votes < 50  THEN 'Ultra Hidden'
        WHEN rating >= 4.5 AND votes < 100 THEN 'Hidden Gem'
        WHEN rating >= 4.0 AND votes < 100 THEN 'Under-the-Radar'
    END AS gem_category
FROM restaurants
WHERE rating >= 4.0 AND votes < 100 AND votes > 0
ORDER BY rating DESC, votes ASC
LIMIT 25
''', 'Q13: Hidden Gems')

q13

In [ ]:
# -------------------------------------------------------
# Q14: Cuisine co-occurrence (Self-Join)
# SQL: Self-join with inequality to avoid duplicates
# -------------------------------------------------------
q14 = run_query('''
SELECT
    c1.cuisine AS cuisine_1,
    c2.cuisine AS cuisine_2,
    COUNT(*) AS co_occurrence_count
FROM restaurant_cuisines c1
JOIN restaurant_cuisines c2
    ON c1.restaurant_id = c2.restaurant_id
    AND c1.cuisine < c2.cuisine
GROUP BY c1.cuisine, c2.cuisine
HAVING COUNT(*) >= 50
ORDER BY co_occurrence_count DESC
LIMIT 20
''', 'Q14: Cuisine Co-occurrence (Self-Join)')

q14

In [ ]:
# -------------------------------------------------------
# Q15: Price segmentation using NTILE(4)
# SQL: NTILE window function + CTE
# -------------------------------------------------------
q15 = run_query('''
WITH price_segments AS (
    SELECT
        id, name, location, rating, approx_cost,
        NTILE(4) OVER (ORDER BY approx_cost) AS price_quartile
    FROM restaurants
    WHERE approx_cost IS NOT NULL
)
SELECT
    CASE price_quartile
        WHEN 1 THEN 'Budget'
        WHEN 2 THEN 'Mid-Range'
        WHEN 3 THEN 'Premium'
        WHEN 4 THEN 'Luxury'
    END AS segment,
    COUNT(*) AS restaurant_count,
    MIN(approx_cost) AS min_cost,
    MAX(approx_cost) AS max_cost,
    ROUND(AVG(approx_cost), 0) AS avg_cost,
    ROUND(AVG(rating), 2) AS avg_rating
FROM price_segments
GROUP BY price_quartile
ORDER BY price_quartile
''', 'Q15: Price Segmentation (NTILE)')

q15

In [ ]:
# ============================================================
# Close connection
# ============================================================
conn.close()
print('\nAll 15 queries ran without errors.')
print('Done with SQL analysis. Head to Notebook 3 for the visualizations.')